## Create the traffic simulator class

Here we will be running a simulation of a click / no click that a user on a site may go through. For this we will first start by defining the traffic simulator class which stores the simulated CTR rates calculated in the previous Python notebook.

In [1]:
import numpy as np
class TrafficSimulator:

    def __init__(self, true_ctrs, seed=42):

        # store the simulated ctr rates of each variant in dictionary
        self.true_ctrs = true_ctrs

        # for reproducability create rng
        self.rng = np.random.default_rng(seed)

    def simulate_click(self, variant : str) -> int:

        # if not a valid variant return
        if variant not in self.true_ctrs:
            print('This variant does not exist.')
            return -1
        
        # get the random number 
        random_no = self.rng.random()

        # returns 1 is lower than p else 0
        return int(random_no < self.true_ctrs[variant])

    def simulate_batch(self, variant: str, n: int) -> int:

        if variant not in self.true_ctrs:
            print('This variant does not exist.')
            return -1

        # return result
        return int (self.rng.random(n) < self.true_ctrs[variant])


In [2]:
true_ctrs = {'A' : 0.657, 'B': 0.2120, 'C' : 0.2167}

In [10]:
import statsmodels.api as sm
import math

# conduct power analysis to see how many visits we want per variant

# calculate the effect size between A vs. B and B vs. C
effect_a_b = sm.stats.proportion_effectsize(true_ctrs['A'], true_ctrs['B'])
effect_b_c = sm.stats.proportion_effectsize(true_ctrs['B'], true_ctrs['C'])

nind_power = sm.stats.NormalIndPower()
# solve for the sample size for the two effect sizes
sample_a_b = math.ceil(nind_power.solve_power(effect_a_b, alpha=0.05, power=0.8))
sample_b_c = math.ceil(nind_power.solve_power(effect_b_c, alpha=0.05, power=0.8))

print(f'A vs. B Sample Size {sample_a_b}')
print(f'B vs. C Sample Size {sample_b_c}')

A vs. B Sample Size 19
B vs. C Sample Size 119670


In [12]:
# calculate the sample size for the simulation for testing
# use a MDE (minimum detectable effect of 10% and base ctr of 0.21 (around the B and C ctrs))

effect_size = sm.stats.proportion_effectsize(0.21, 0.231)
sample_size = math.ceil(nind_power.solve_power(effect_size, alpha=0.05, power=0.8))
print(f'Sample Size {sample_size}')

Sample Size 6116
